# Project 1 - Part 3: Deep Learning

This notebook contains the image classification project development process using deep learning algorithms.

## Project Goals:
- Prepare a 3-class image dataset (cup, pen, keyboard)
- Create Convolutional Neural Network (CNN) model
- Perform model training and evaluation
- Apply transfer learning
- Optimize and improve model performance

## Dataset Structure:
```
proje_veri_seti/
├── bardak/
│   ├── 1.jpg
│   ├── 2.jpg
│   └── ...
├── kalem/
│   ├── 1.jpg
│   ├── 2.jpg
│   └── ...
└── klavye/
    ├── 1.jpg
    ├── 2.jpg
    └── ...
```

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
import matplotlib.pyplot as plt
import pathlib

# Gerekli kütüphaneleri kurun:
# pip install tensorflow

# --- Veri Seti Hazırlığı ---

# Veri setinizin yolunu belirtin (KENDİNİZE GÖRE DEĞİŞTİRİN)
data_dir = pathlib.Path("proje_veri_seti") 

# Görüntü parametreleri
batch_size = 32 # Veri seti küçük olduğu için daha küçük olabilir, örn: 8
img_height = 180
img_width = 180
IMG_SIZE = (img_height, img_width)

# Veriyi Eğitim ve Validasyon olarak ayırma
# Toplam 30 görüntümüz olduğu için %20'si (6 adet) validasyon olacak.
train_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=IMG_SIZE,
  batch_size=batch_size)

val_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=IMG_SIZE,
  batch_size=batch_size)

# Sınıf isimlerini al
class_names = train_ds.class_names
num_classes = len(class_names)
print(f"Bulunan Sınıflar: {class_names}")

# Veri Artırma (Data Augmentation) - Veri seti küçük olduğu için ŞART!
data_augmentation = keras.Sequential(
  [
    layers.RandomFlip("horizontal", input_shape=(img_height, img_width, 3)),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
  ]
)

# Performans için veri setlerini önbelleğe al
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# Görüntüleri [0, 1] aralığına normalize et
normalization_layer = layers.Rescaling(1./255)

# --- Model 1: Sıfırdan Basit bir CNN Modeli ---
# Rapor Adı: Basit Evrişimli Sinir Ağı (CNN)
# Özellikler: 3 adet Conv2D katmanı ve 2 adet Dense katmandan oluşan, 
# veri artırma kullanan basit bir model.
model1 = Sequential([
  data_augmentation,
  normalization_layer,
  layers.Conv2D(16, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(32, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(64, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Flatten(),
  layers.Dense(128, activation='relu'),
  layers.Dense(num_classes, activation='softmax') # Son katman (sınıf sayısı kadar nöron)
])

model1.compile(optimizer='adam',
               loss=tf.keras.losses.SparseCategoricalCrossentropy(),
               metrics=['accuracy'])

print("\n--- Model 1 (Basit CNN) Eğitimi Başlıyor ---")
history1 = model1.fit(
  train_ds,
  validation_data=val_ds,
  epochs=15 # Az veri olduğu için çok uzun eğitmiyoruz
)
print("Model 1 Eğitimi Bitti.")

# --- Model 2: Transfer Öğrenme (MobileNetV2) ---
# Rapor Adı: MobileNetV2 ile Transfer Öğrenme
# Özellikler: ImageNet üzerinde eğitilmiş MobileNetV2 modelinin evrişim 
# katmanları (base model) alınmış, dondurulmuş ve üzerine yeni bir 
# sınıflandırıcı kafa (classifier head) eklenmiştir.

# Girdi boyutunu MobileNetV2'nin beklediği 3 kanala (RGB) ayarlıyoruz
# (Veri setimiz zaten RGB olmalı)
IMG_SHAPE = IMG_SIZE + (3,)

# Önceden eğitilmiş modeli yükle (sınıflandırıcı katmanı olmadan)
base_model = tf.keras.applications.MobileNetV2(input_shape=IMG_SHAPE,
                                               include_top=False,
                                               weights='imagenet')

# Önceden eğitilmiş katmanları dondur (tekrar eğitilmesinler)
base_model.trainable = False

# Veri artırma ve normalizasyonu modele dahil et
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input
rescale = tf.keras.layers.Rescaling(1./127.5, offset=-1) # MobileNet [-1, 1] bekler

model2 = Sequential([
    data_augmentation,
    #preprocess_input, # Alternatif olarak bu kullanılabilir, ancak rescaling daha geneldir
    rescale,
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(num_classes, activation='softmax')
])

model2.compile(optimizer='adam',
               loss=tf.keras.losses.SparseCategoricalCrossentropy(),
               metrics=['accuracy'])

print("\n--- Model 2 (Transfer Öğrenme) Eğitimi Başlıyor ---")
history2 = model2.fit(
  train_ds,
  validation_data=val_ds,
  epochs=15
)
print("Model 2 Eğitimi Bitti.")

# --- Değerlendirme ---
print("\n--- MODELLERİN DEĞERLENDİRMESİ (Test/Validasyon Verisi) ---")
loss1, acc1 = model1.evaluate(val_ds)
loss2, acc2 = model2.evaluate(val_ds)

print(f"Model 1 (Basit CNN) Doğruluk: {acc1*100:.2f}%")
print(f"Model 2 (Transfer Öğrenme) Doğruluk: {acc2*100:.2f}%")

## Eğitim Sonuçlarını Görselleştirme

In [ ]:
# Eğitim sonuçlarını görselleştir
def plot_history(history, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Doğruluk grafiği
    ax1.plot(history.history['accuracy'], label='Eğitim Doğruluğu')
    ax1.plot(history.history['val_accuracy'], label='Validasyon Doğruluğu')
    ax1.set_title(f'{title} - Doğruluk')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Doğruluk')
    ax1.legend()
    ax1.grid(True)
    
    # Kayıp grafiği
    ax2.plot(history.history['loss'], label='Eğitim Kaybı')
    ax2.plot(history.history['val_loss'], label='Validasyon Kaybı')
    ax2.set_title(f'{title} - Kayıp')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Kayıp')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()

# Model 1 sonuçları
print("Model 1 (Basit CNN) Eğitim Grafikleri:")
plot_history(history1, 'Basit CNN')

# Model 2 sonuçları
print("\nModel 2 (Transfer Öğrenme) Eğitim Grafikleri:")
plot_history(history2, 'Transfer Öğrenme - MobileNetV2')

## Model Karşılaştırması ve Sonuçlar

In [ ]:
# Model performanslarını karşılaştır
print("=" * 60)
print("MODEL KARŞILAŞTIRMASI")
print("=" * 60)
print(f"\nModel 1 - Basit CNN:")
print(f"  Validasyon Doğruluğu: {acc1*100:.2f}%")
print(f"  Validasyon Kaybı: {loss1:.4f}")

print(f"\nModel 2 - Transfer Öğrenme (MobileNetV2):")
print(f"  Validasyon Doğruluğu: {acc2*100:.2f}%")
print(f"  Validasyon Kaybı: {loss2:.4f}")

print(f"\nPerformans Farkı:")
if acc2 > acc1:
    print(f"  Transfer Öğrenme, Basit CNN'den {(acc2-acc1)*100:.2f}% daha iyi!")
else:
    print(f"  Basit CNN, Transfer Öğrenme'den {(acc1-acc2)*100:.2f}% daha iyi!")

print("=" * 60)

# Karşılaştırma grafiği
plt.figure(figsize=(10, 6))
models = ['Basit CNN', 'Transfer Öğrenme\n(MobileNetV2)']
accuracies = [acc1*100, acc2*100]

bars = plt.bar(models, accuracies, color=['#3498db', '#e74c3c'], alpha=0.8, edgecolor='black', linewidth=2)
plt.ylabel('Doğruluk (%)', fontsize=12)
plt.title('Model Performans Karşılaştırması', fontsize=14, fontweight='bold')
plt.ylim(0, 100)
plt.grid(axis='y', alpha=0.3)

# Bar üzerine yüzde değerlerini yaz
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
            f'{acc:.2f}%',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## Örnek Tahmin Yapma

Eğitilmiş modellerle veri setinden rastgele görüntüler üzerinde tahmin yapalım.

In [ ]:
# Veri setinden örnek tahminler yap
import numpy as np

# Validasyon setinden bir batch al
for images, labels in val_ds.take(1):
    # İlk 6 görüntüyü göster
    plt.figure(figsize=(15, 10))
    
    for i in range(min(6, len(images))):
        # Model 1 ile tahmin
        pred1 = model1.predict(tf.expand_dims(images[i], 0), verbose=0)
        pred1_class = class_names[np.argmax(pred1[0])]
        pred1_conf = np.max(pred1[0]) * 100
        
        # Model 2 ile tahmin
        pred2 = model2.predict(tf.expand_dims(images[i], 0), verbose=0)
        pred2_class = class_names[np.argmax(pred2[0])]
        pred2_conf = np.max(pred2[0]) * 100
        
        # Gerçek sınıf
        true_class = class_names[labels[i]]
        
        # Görüntüyü göster
        plt.subplot(2, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(f'Gerçek: {true_class}\nCNN: {pred1_class} ({pred1_conf:.1f}%)\nTransfer: {pred2_class} ({pred2_conf:.1f}%)',
                 fontsize=9)
        plt.axis('off')
    
    plt.tight_layout()
    plt.show()
    break

## Modelleri Kaydetme

Eğitilmiş modelleri sonradan kullanmak üzere kaydedelim.

In [ ]:
# Modelleri kaydet
model1.save('basit_cnn_model.h5')
print("✅ Model 1 'basit_cnn_model.h5' olarak kaydedildi.")

model2.save('transfer_learning_model.h5')
print("✅ Model 2 'transfer_learning_model.h5' olarak kaydedildi.")

print("\n📌 Modelleri yüklemek için:")
print("   loaded_model = tf.keras.models.load_model('basit_cnn_model.h5')")

## 📊 Sonuçlar ve Değerlendirme

### Proje Özeti:
- **Veri Seti**: 3 sınıflı görüntü sınıflandırma (filizlenme, olgunlaşma, kış uykusu dönemleri)
- **Model 1**: Sıfırdan eğitilen basit CNN modeli
- **Model 2**: MobileNetV2 ile Transfer Öğrenme
- **Veri Artırma**: Rastgele döndürme, kaydırma ve zoom uygulandı

### Önemli Bulgular:
1. **Transfer Öğrenme Avantajı**: Önceden eğitilmiş modeller, küçük veri setlerinde daha iyi performans gösterir
2. **Veri Artırmanın Önemi**: Az veriyle overfitting'i önlemek için kritik
3. **Model Karmaşıklığı**: Daha karmaşık model her zaman daha iyi sonuç vermeyebilir

### İyileştirme Önerileri:

#### 1. Veri Seti İyileştirmeleri:
- Her sınıf için daha fazla görüntü toplayın (100-500+ görüntü)
- Farklı açılardan ve ışık koşullarında çekim yapın
- Görüntü kalitesine dikkat edin
- Veri dengesini sağlayın

#### 2. Model İyileştirmeleri:
- Fine-tuning: Transfer learning modelinin üst katmanlarını da eğitin
- Farklı mimariler deneyin (ResNet, EfficientNet)
- Ensemble yöntemleri kullanın
- Hiperparametre optimizasyonu yapın

#### 3. İleri Seviye Teknikler:
- Cross-validation uygulayın
- Learning rate scheduling
- Advanced data augmentation (Mixup, CutMix)
- Attention mekanizmaları

---

### 🚀 Sonraki Adımlar:
1. Kendi veri setinizi hazırlayın
2. Modelleri kendi verinizle eğitin
3. Sonuçları analiz edin ve rapor hazırlayın
4. Gerçek zamanlı tahmin uygulaması geliştirin

**Başarılar! 🎉**